# Лабораторная работа №5 — генеративные модели изображений

**Дисциплина:** Интеллектуальный анализ данных  
**Выполнил:** _(ФИО, группа)_  
**Дата:** _(дата)_

## Цели
1. Реализовать **нейросетевой перенос стиля** (Gatys et al.) на основе предобученной VGG.
2. Обучить **DCGAN** на наборе **EMNIST (digits)** и проанализировать качество генерации и интерполяцию в латентном пространстве.

> В Kaggle: **Settings → Accelerator → GPU P100/T4** (рекомендуется для ускорения). Включите **Internet**, если нужна загрузка весов/dataset.

---
## Часть 1. Neural Style Transfer (NST)

Идея: совместно минимизировать потери **содержания** (на карте признаков VGG) и **стиля** (через матрицы Грама по нескольким слоям). Оптимизируется само изображение (пиксели), а не веса сети.

In [ ]:
# Ячейка 1 — окружение и устройство
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models, datasets
from torchvision.utils import save_image, make_grid
from PIL import Image

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# Результаты: на Kaggle — /kaggle/working; локально — ./lab5_output
OUT = "/kaggle/working/lab5" if os.path.isdir("/kaggle/working") else "lab5_output"
os.makedirs(f"{OUT}/nst", exist_ok=True)
os.makedirs(f"{OUT}/gan", exist_ok=True)

In [ ]:
# Ячейка 2 — загрузка изображений и нормализация под ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def load_image(path, size=256):
    image = Image.open(path).convert("RGB")
    t = transforms.Compose(
        [
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )
    return t(image).unsqueeze(0)


def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = tensor.clone().squeeze(0).cpu()
    img = img * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


def tensor_to_show(t):
    """Тензор в RGB [0,1] для matplotlib (без нормализации ImageNet)."""
    mean = torch.tensor(IMAGENET_MEAN, device=t.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=t.device).view(1, 3, 1, 1)
    x = (t * std + mean).clamp(0, 1)
    return x.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()

In [ ]:
# Ячейка 3 — VGG19, слои стиля/контента, функции потерь
print("Загрузка VGG19...")
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device)
for p in vgg.parameters():
    p.requires_grad = False
vgg.eval()

style_layers = {"conv1_1": 0, "conv2_1": 5, "conv3_1": 10, "conv4_1": 19, "conv5_1": 28}
content_layers = {"conv4_2": 21}
all_layers = {**style_layers, **content_layers}


def get_features(image, model, layers):
    idx_to_name = {v: k for k, v in layers.items()}
    max_idx = max(idx_to_name.keys())
    feats = {}
    x = image
    for i, layer in enumerate(model):
        x = layer(x)
        if i in idx_to_name:
            feats[idx_to_name[i]] = x
        if i == max_idx:
            break
    return feats


def content_loss(gen_f, content_f):
    return F.mse_loss(gen_f, content_f)


def gram_matrix(f):
    b, c, h, w = f.shape
    Fm = f.view(b, c, h * w)
    G = torch.bmm(Fm, Fm.transpose(1, 2))
    return G / (c * h * w)


def style_loss(gen_f, style_f):
    return F.mse_loss(gram_matrix(gen_f), gram_matrix(style_f))

In [ ]:
# Ячейка 4 — один прогон переноса стиля (LBFGS по пикселям)
def run_style_transfer(content_path, style_path, out_path, img_size=256, num_steps=300, alpha=1.0, beta=1e5, init_from_content=True):
    content_img = load_image(content_path, size=img_size).to(device)
    style_img = load_image(style_path, size=img_size).to(device)

    if init_from_content:
        generated = content_img.clone().requires_grad_(True)
    else:
        generated = torch.randn_like(content_img).requires_grad_(True)

    with torch.no_grad():
        content_features = get_features(content_img, vgg, all_layers)
        style_features = get_features(style_img, vgg, all_layers)

    optimizer = optim.LBFGS([generated], lr=1.0, max_iter=20)
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)
    last_losses = {}

    def closure():
        optimizer.zero_grad()
        gen_f = get_features(generated, vgg, all_layers)
        c_loss = content_loss(gen_f["conv4_2"], content_features["conv4_2"])
        s_loss = sum(style_loss(gen_f[l], style_features[l]) for l in style_layers)
        total = alpha * c_loss + beta * s_loss
        total.backward()
        last_losses["total"] = total.item()
        last_losses["c"] = c_loss.item()
        last_losses["s"] = s_loss.item()
        return total

    for step in range(num_steps):
        optimizer.step(closure)
        with torch.no_grad():
            generated.data = (generated.data * std + mean).clamp(0, 1)
            generated.data = (generated.data - mean) / std
        if step % 50 == 0 or step == num_steps - 1:
            print(f"  step {step:4d} | total={last_losses['total']:.4f} | content={last_losses['c']:.4f} | style={last_losses['s']:.4f}")

    result = (generated.data * std + mean).clamp(0, 1)
    save_image(result, out_path)
    return generated

In [ ]:
# Ячейка 5 — подготовка content/style
# Вариант A: положите свои файлы в Kaggle Dataset и подключите его как Input.
# Тогда задайте пути, например:
# CONTENT = "/kaggle/input/your-dataset/content.jpg"
# STYLE = "/kaggle/input/your-dataset/style.jpg"

# Вариант B (без своих картинок): синтетические тестовые изображения
data_dir = f"{OUT}/data_nst"
os.makedirs(data_dir, exist_ok=True)
CONTENT = os.path.join(data_dir, "content.png")
STYLE = os.path.join(data_dir, "style.png")

if not (os.path.isfile(CONTENT) and os.path.isfile(STYLE)):
    c = np.zeros((256, 256, 3), dtype=np.uint8)
    c[64:192, 64:192] = [255, 200, 100]
    Image.fromarray(c).save(CONTENT)
    s = np.zeros((256, 256, 3), dtype=np.uint8)
    for i in range(0, 256, 32):
        for j in range(0, 256, 32):
            s[i : i + 32, j : j + 32] = [200, 50, 50] if (i // 32 + j // 32) % 2 == 0 else [50, 50, 200]
    Image.fromarray(s).save(STYLE)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(Image.open(CONTENT))
ax[0].set_title("Content")
ax[0].axis("off")
ax[1].imshow(Image.open(STYLE))
ax[1].set_title("Style")
ax[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Ячейка 6 — эксперименты NST: вес стиля β и инициализация
experiments = [
    ("beta=1e5 (базовый)", 1e5, True),
    ("beta=1e3 (слабый стиль)", 1e3, True),
    ("beta=1e7 (сильный стиль)", 1e7, True),
]

results_nst = []
for name, beta, from_content in experiments:
    print("\n", name)
    path = f"{OUT}/nst/result_{beta}.png"
    run_style_transfer(CONTENT, STYLE, path, img_size=256, num_steps=300, alpha=1.0, beta=beta, init_from_content=from_content)
    results_nst.append((name, path))

print("\nИнициализация шумом (beta=1e5)")
path_noise = f"{OUT}/nst/result_noise_init.png"
run_style_transfer(CONTENT, STYLE, path_noise, num_steps=300, alpha=1.0, beta=1e5, init_from_content=False)
results_nst.append(("шум", path_noise))

fig, axes = plt.subplots(1, len(results_nst), figsize=(4 * len(results_nst), 4))
for ax, (title, p) in zip(np.atleast_1d(axes), results_nst):
    ax.imshow(Image.open(p))
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.suptitle("NST: сравнение")
plt.tight_layout()
plt.show()

### Краткие выводы по части 1 *(заполните после запуска)*
- Как влияет увеличение $\beta$: ...
- Чем отличается результат при инициализации от шума: ...

---
## Часть 2. DCGAN на EMNIST (digits)

Генератор переводит шум $z \sim \mathcal{N}(0,I)$ в изображение $32\times32$; дискриминатор отличает реальные цифры от сгенерированных. Архитектура — типичная DCGAN (Radford et al.).

In [ ]:
# Ячейка 7 — данные EMNIST
transform = transforms.Compose(
    [
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

dataset = datasets.EMNIST(root=f"{OUT}/data", split="digits", train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())

real_batch, _ = next(iter(dataloader))
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Примеры EMNIST digits (нормализованы в [-1,1])")
plt.imshow(np.transpose(make_grid(real_batch[:64], nrow=8, normalize=True, value_range=(-1, 1)).cpu(), (1, 2, 0)))
plt.show()

In [ ]:
# Ячейка 8 — архитектура G и D
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0, bias=False),
        )

    def forward(self, img):
        return self.net(img).view(-1)


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


latent_dim = 100
G = Generator(latent_dim=latent_dim, img_channels=1).to(device)
D = Discriminator(img_channels=1).to(device)
G.apply(weights_init)
D.apply(weights_init)
print("Параметры G:", sum(p.numel() for p in G.parameters()))
print("Параметры D:", sum(p.numel() for p in D.parameters()))

In [ ]:
# Ячейка 9 — обучение (при необходимости уменьшите num_epochs для теста)
criterion = nn.BCEWithLogitsLoss()
lr = 2e-4
opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, latent_dim, 1, 1, device=device)
num_epochs = 15
history = {"loss_D": [], "loss_G": [], "D_x": [], "D_G_z": []}

for epoch in range(num_epochs):
    epoch_loss_D = epoch_loss_G = epoch_D_x = epoch_D_G_z = 0.0
    n_batches = 0
    for real_images, _ in dataloader:
        bs = real_images.size(0)
        real_images = real_images.to(device)
        real_labels = torch.ones(bs, device=device)
        fake_labels = torch.zeros(bs, device=device)

        # D
        z = torch.randn(bs, latent_dim, 1, 1, device=device)
        fake_images = G(z)
        d_real = D(real_images)
        d_fake = D(fake_images.detach())
        loss_D = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # G
        z = torch.randn(bs, latent_dim, 1, 1, device=device)
        fake_images = G(z)
        d_fake_for_G = D(fake_images)
        loss_G = criterion(d_fake_for_G, torch.ones(bs, device=device))
        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        epoch_loss_D += loss_D.item()
        epoch_loss_G += loss_G.item()
        epoch_D_x += torch.sigmoid(d_real).mean().item()
        epoch_D_G_z += torch.sigmoid(d_fake_for_G).mean().item()
        n_batches += 1

    history["loss_D"].append(epoch_loss_D / n_batches)
    history["loss_G"].append(epoch_loss_G / n_batches)
    history["D_x"].append(epoch_D_x / n_batches)
    history["D_G_z"].append(epoch_D_G_z / n_batches)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | loss_D={history['loss_D'][-1]:.4f} | loss_G={history['loss_G'][-1]:.4f} | "
        f"D(x)={history['D_x'][-1]:.3f} | D(G(z))={history['D_G_z'][-1]:.3f}"
    )

    if (epoch + 1) % 5 == 0 or epoch == 0:
        with torch.no_grad():
            fake = G(fixed_noise)
            save_image(fake, f"{OUT}/gan/epoch_{epoch+1:03d}.png", nrow=8, normalize=True, value_range=(-1, 1))

In [ ]:
# Ячейка 10 — графики, сетка, интерполяция, сохранение
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["loss_D"], label="D")
axes[0].plot(history["loss_G"], label="G")
axes[0].legend()
axes[0].set_title("Потери")
axes[0].grid(True, alpha=0.3)
axes[1].plot(history["D_x"], label="D(x)")
axes[1].plot(history["D_G_z"], label="D(G(z))")
axes[1].axhline(0.5, color="k", ls="--", alpha=0.4)
axes[1].legend()
axes[1].set_title("Уверенность дискриминатора")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/gan/history.png", dpi=150)
plt.show()

G.eval()
with torch.no_grad():
    grid = G(torch.randn(64, latent_dim, 1, 1, device=device))
    save_image(grid, f"{OUT}/gan/final_grid.png", nrow=8, normalize=True, value_range=(-1, 1))

z1 = torch.randn(1, latent_dim, 1, 1, device=device)
z2 = torch.randn(1, latent_dim, 1, 1, device=device)
alphas = torch.linspace(0, 1, steps=10)
with torch.no_grad():
    interp = torch.cat([(1 - a) * z1 + a * z2 for a in alphas], dim=0)
    imgs = G(interp)
    save_image(imgs, f"{OUT}/gan/interpolation.png", nrow=10, normalize=True, value_range=(-1, 1))

plt.figure(figsize=(12, 2))
plt.imshow(np.transpose(make_grid(imgs, nrow=10, normalize=True, value_range=(-1, 1)).cpu(), (1, 2, 0)))
plt.axis("off")
plt.title("Интерполяция между двумя точками в Z")
plt.show()

torch.save(
    {
        "G": G.state_dict(),
        "D": D.state_dict(),
        "history": history,
        "latent_dim": latent_dim,
    },
    f"{OUT}/gan/dcgan_emnist.pth",
)
print("Сохранено:", f"{OUT}/gan/dcgan_emnist.pth")

### Краткие выводы по части 2 *(заполните)*
- Наблюдения по траектории потерь и стабильности обучения: ...
- Качество сгенерированных цифр, типичные артефакты: ...
- Что показывает линейная интерполяция в $z$: ...